In [9]:
import pandas as pd
import json
import ast
import warnings
import dask.dataframe as dd
warnings.filterwarnings("ignore",category=Warning)
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore",category=PerformanceWarning)

In [ ]:
df = pd.read_csv('')

In [7]:
import pandas as pd
import json
import ast


def build_df_new(df, key_cols=("idcard", "dateBack"), card_col="card"):
    """
    按以下规则从原始 df 生成新的 df_new，不修改原 df：

    1. 主键: idcard + dateBack
    2. 所有 *_max 字段:
       - 同一 idcard/dateBack 下多行 json 合并
       - 输出为下游可直接 json.loads() 解析的合法 JSON 字符串
    3. 所有 *_fail_out_max 字段:
       - 基于合并后的 *_fail_out_max 生成 *_fail_out_amt 和 *_fail_out_cnt
    4. 其他 *_amt / *_cnt 字段:
       - 排除 *_fail_out_amt / *_fail_out_cnt
       - 按 idcard/dateBack 求和
    5. cardnum:
       - 按同一 idcard 统计不同 card 数量
       - 不区分 dateBack
    """

    df_src = df.copy(deep=True)
    key_cols = list(key_cols)

    for col in key_cols:
        if col not in df_src.columns:
            raise KeyError(f"主键列不存在: {col}")

    if card_col not in df_src.columns:
        raise KeyError(f"卡号列不存在: {card_col}")
    print('A')

    def _to_number(v):
        """尽量把值转成数值；转不了返回 None。"""
        if v is None:
            return None
        if isinstance(v, bool):
            return int(v)
        if isinstance(v, (int, float)):
            if pd.isna(v):
                return None
            return int(v) if isinstance(v, float) and v.is_integer() else v
        if isinstance(v, str):
            s = v.strip()
            if not s:
                return None
            try:
                num = float(s)
                return int(num) if num.is_integer() else num
            except Exception:
                return None
        return None

    def _parse_json_like(x):
        """
        把 max 字段里的内容转成 dict。
        支持:
        - dict
        - 合法 json 字符串
        - 少数 python dict 字符串格式（做兜底）
        """
        if x is None:
            return {}
        if isinstance(x, float) and pd.isna(x):
            return {}
        if isinstance(x, dict):
            obj = x
        elif isinstance(x, str):
            s = x.strip()
            if not s or s == "{}":
                return {}
            try:
                obj = json.loads(s)
            except Exception:
                try:
                    obj = ast.literal_eval(s)
                except Exception:
                    return {}
        else:
            return {}

        if not isinstance(obj, dict):
            return {}

        out = {}
        for k, v in obj.items():
            num = _to_number(v)
            out[str(k)] = num if num is not None else v
        return out

    def _merge_json_series(series):
        """
        把同组内多行 json 合并成一个 dict，再转回合法 JSON 字符串。
        注意:
        - 如果不同记录里出现相同 key，后面的值会覆盖前面的值
        """
        merged = {}
        for x in series:
            d = _parse_json_like(x)
            merged.update(d)

        return json.dumps(merged, ensure_ascii=False, separators=(",", ":"))

    def _sum_json_values(x):
        """对 json 字典中的 value 求和。"""
        d = _parse_json_like(x)
        total = 0
        for v in d.values():
            num = _to_number(v)
            if num is not None:
                total += num
        return int(total) if isinstance(total, float) and total.is_integer() else total

    def _count_json_values(x):
        """统计 json 字典中数值型 value 的个数。"""
        d = _parse_json_like(x)
        cnt = 0
        for v in d.values():
            if _to_number(v) is not None:
                cnt += 1
        return cnt

    # 1. 先生成主键表，不改原表
    df_new = df_src[key_cols].drop_duplicates().reset_index(drop=True)
    print('B')

    # 2. 合并所有 *_max 字段
    max_cols = [c for c in df_src.columns if c.endswith("_max")]
    if max_cols:
        max_agg = (
            df_src.groupby(key_cols, dropna=False)[max_cols]
            .agg(_merge_json_series)
            .reset_index()
        )
        df_new = df_new.merge(max_agg, on=key_cols, how="left")
    print('C')

    # 3. 从合并后的 *_fail_out_max 生成 *_fail_out_amt / *_fail_out_cnt
    fail_out_max_cols = [c for c in max_cols if c.endswith("_fail_out_max")]
    generated_fail_cols = []

    for col in fail_out_max_cols:
        base_name = col[:-4]  # 去掉 "_max"
        amt_col = f"{base_name}_amt"
        cnt_col = f"{base_name}_cnt"

        df_new[amt_col] = df_new[col].apply(_sum_json_values)
        df_new[cnt_col] = df_new[col].apply(_count_json_values)

        generated_fail_cols.extend([amt_col, cnt_col])
    print('D')

    # 4. 处理其他 amt/cnt 字段
    #    排除 fail_out_amt / fail_out_cnt，因为这两个必须从 fail_out_max 重新生成
    other_sum_cols = [
        c
        for c in df_src.columns
        if (c.endswith("_amt") or c.endswith("_cnt"))
        and not c.endswith("_fail_out_amt")
        and not c.endswith("_fail_out_cnt")
    ]

    if other_sum_cols:
        tmp = df_src[key_cols + other_sum_cols].copy()
        for c in other_sum_cols:
            tmp[c] = pd.to_numeric(tmp[c], errors="coerce").fillna(0)

        other_agg = (
            tmp.groupby(key_cols, dropna=False, as_index=False)[other_sum_cols]
            .sum()
        )
        df_new = df_new.merge(other_agg, on=key_cols, how="left")
    print('E')

    # 5. 计算 cardnum: 同一 idcard 下不同 card 数量，不分 dateBack
    cardnum_df = (
        df_src.groupby("idcard", dropna=False)[card_col]
        .nunique(dropna=True)
        .reset_index(name="cardnum")
    )
    df_new = df_new.merge(cardnum_df, on="idcard", how="left")
    print('F')

    # 6. 调整列顺序
    ordered_cols = key_cols + ["cardnum"] + max_cols + generated_fail_cols + other_sum_cols
    ordered_cols = [c for c in dict.fromkeys(ordered_cols) if c in df_new.columns]
    remain_cols = [c for c in df_new.columns if c not in ordered_cols]
    df_new = df_new[ordered_cols + remain_cols]
    print('G')

    #7.加index
    df_new['index'] = df_new.groupby(['idcard','dateBack'],sort=False).ngroup()

    return df_new

In [ ]:
df_new = build_df_new(df, key_cols=("idcard", "dateBack"), card_col="card")

In [ ]:
def get(df,file_name):
    cols_max = [i for i in df.columns if i.endswith('_max')]
    df[cols_max] = df[cols_max].applymap(lambda x : json.loads(x))

    #多个字典合并，取最大值对应的机构，返回dict
    def dict_max_dict(ds):    
        data_res = dict()
    
        for i in ds:
            s = data_res.keys()&(i.keys())
            if len(s) == 0:
                data_res.update(i)
            else:
                for key,value in i.items():
                    if key in s:
                        data_res[key] = max(data_res[key], value)
                    else:
                        data_res[key] = value
        return data_res
    
    #多个字典，取最大值，返回数值
    def dict_max_num(ds):
        
        data_res = 0
        
        for i in ds:
            if len(i) == 0:
                continue
            else:
                s = max(i.values())
                data_res = max(s,data_res)
        return data_res
    
    #字典取最大值,返回数值
    def dict_max(dct):
        if len(dct) == 0:
            return 0
        else:
            return max(dct.values())
    
    def new(s1,s2):
        a = set(s1.keys())-set(s2.keys())
        la = len(a)
        b = {key: s1[key] for key in a}
        return la,b
        
    def new_org(f1,f2):
        df = pd.concat([f1,f2],axis = 1)
        df.columns = ['font','back']
        df['dif'] = df.apply(lambda x : new(x['font'],x['back'])[0],axis = 1)
        df['new_max'] = df.apply(lambda x : new(x['font'],x['back'])[1],axis = 1)
        
        
        return df['dif'],df['new_max']
    
    #两列相除
    def div_(x):
        if x[1] == 0:
            return 0
        else:
            return x[0]/x[1]
    def dif_ser(f1,f2):
        f = pd.concat([f1,f2],axis = 1)
        f.columns = [0,1]
        return f.apply(div_,axis = 1)
    
    #查找第一个非0
    def find_non_zero(f):
        lst = list(f)
        try:
            return next(i for i, x in enumerate(lst) if x != 0)
        except StopIteration:
            return 99999 
    #由后到前查找第一个非0
    def find_non_zero_back(f):
        lst = list(f)[::-1]
        lth = len(lst)-1
        try:
            return lth - next(i for i, x in enumerate(lst) if x != 0)
        except StopIteration:
            return 99999 
    
    #dt_card_ = df[['idcard','card']].drop_duplicates().groupby(['idcard']).agg({'card':'count'})
    #dt_card_.reset_index(inplace = True)
    dt_card_ = df[['idcard','cardnum']]
    print('A')
    
    #金额、笔数聚合
    cols_cnt = [i for i in df.columns if i.endswith('_cnt')]
    cols_amt = [i for i in df.columns if i.endswith('_amt')]
    dt_amt_cnt = df[['index'] + cols_cnt + cols_amt].groupby(['index']).sum()
    
    #max聚合
    cols_max = [i for i in df.columns if i.endswith('_max')]
    dt_max_ = pd.DataFrame()
    dt_max_grouped = df[['index'] + cols_max].groupby(['index'])
    for i in cols_max:
        dt_max_f = dt_max_grouped.agg({i:dict_max_dict})
        dt_max_ = pd.concat([dt_max_,dt_max_f],axis = 1)
    
    da = pd.concat([dt_amt_cnt,dt_max_],axis = 1)
    da.reset_index(inplace = True)
    
    df_drop = df[['index','idcard','dateBack']].drop_duplicates()
    data_all = pd.merge(df_drop,da,on=['index'],how='right')
    data = dt_card_.merge(data_all,on = ['idcard'],how = 'right')
    print('B')
    
    date = [0,1,2,3,6,12,18,23]
    org_type1 = ['bank','cf', 'top', 'others', 'nonloan']
    org_type2 = ['bank','cf', 'top', 'others']
    state_add = ['suc_out_cnt', 'suc_out_amt', 'suc_out_max', 'fail_out_cnt',
           'fail_out_amt', 'fail_out_max', 'suc_in_cnt', 'suc_in_amt']
    
    #近X月
    dt = pd.DataFrame()
    for i in date:
        for j in state_add:
            if (i == 0)|(i == 1):
                if j in ['suc_out_cnt', 'suc_out_amt']:
                    for k in org_type1:
                        s = str(i) + '_' + k + '_' + j
                        dt['rec_' + s] = data['pre_' + s]
                else:
                    for k in org_type2:
                        s = str(i) + '_' + k + '_' + j
                        dt['rec_' + s] = data['pre_' + s]
            else:
                if j in ['suc_out_cnt', 'suc_out_amt']:                    
                    for k in org_type1: 
                        s = str(i) + '_' + k + '_' + j
                        dt['rec_' + s] = 0
                        for m in range(1,i+1):
                            ss = str(m) + '_' + k + '_' + j
                            dt['rec_' + s] = dt['rec_' + s] + data['pre_' + ss]
                elif j.split('_')[-1] != 'max':
                    for k in org_type2:
                        s = str(i) + '_' + k + '_' + j
                        dt['rec_' + s] = 0
                        for m in range(1,i+1):
                            ss = str(m) + '_' + k + '_' + j
                            dt['rec_' + s] = dt['rec_' + s] + data['pre_' + ss]
                else:
                    for k in org_type2:
                        dt_max = pd.DataFrame()
                        s = str(i) + '_' + k + '_' + j
                        for m in range(1,i+1):
                            ss = str(m) + '_' + k + '_' + j
                            dt_max = pd.concat([dt_max,data['pre_' + ss]],axis = 1)
                        dt['rec_' + s] = dt_max.apply(dict_max_dict,axis = 1)
    
    
    #近X月机构汇总类
    for i in date:
        for j in state_add:
            s = 'rec_' + str(i) + '_all_' + j
            dt[s] = 0
            if j in ['suc_out_cnt', 'suc_out_amt']:
                for k in org_type1:
                    dt[s] = dt[s] + dt['rec_' + str(i) + '_' + k + '_' + j]
            elif j.split('_')[-1] != 'max':
                for k in org_type2:
                    dt[s] = dt[s] + dt['rec_' + str(i) + '_' + k + '_' + j]
            else:
                dt_max = pd.DataFrame()
                for k in org_type2:
                    dt_max = pd.concat([dt_max,dt['rec_' + str(i) + '_' + k + '_' + j]],axis = 1)
                dt[s] = dt_max.apply(dict_max_dict,axis = 1)
    print('C')
    
    
    #按月汇总max,临时使用，不在列表中
    for i in range(24):
        for j in ['suc_out_max', 'fail_out_max']:
            s = 'rec_' + str(i) +'_' + j + '_all'
            dt_max = pd.DataFrame()
            for k in org_type2:
                dt_max = pd.concat([dt_max,data['pre_' + str(i) + '_' + k + '_' + j]],axis = 1)
            dt[s] = dt_max.apply(dict_max_dict,axis = 1)
    
    
    #均值
    org_type3 = ['bank','cf', 'top', 'others','all']
    for i in date:
        for j in org_type3:
            dt['rec_' + str(i) + '_' + j + '_suc_out_avg'] = dif_ser(dt['rec_' + str(i) + '_' + j + '_suc_out_amt'],dt['rec_' + str(i) + '_' + j + '_suc_out_cnt'])
            dt['rec_' + str(i) + '_' + j + '_fail_out_avg'] = dif_ser(dt['rec_' + str(i) + '_' + j + '_fail_out_amt'],dt['rec_' + str(i) + '_' + j + '_fail_out_cnt'])
            dt['rec_' + str(i) + '_' + j + '_suc_in_avg'] = dif_ser(dt['rec_' + str(i) + '_' + j + '_suc_in_amt'],dt['rec_' + str(i) + '_' + j + '_suc_in_cnt'])
        dt['rec_' + str(i) + '_nonloan_suc_out_avg'] = dif_ser(dt['rec_' + str(i) + '_nonloan_suc_out_amt'],dt['rec_' + str(i) + '_nonloan_suc_out_cnt'])
    
    #占比
    for i in date:
        for j in org_type3:
            t1 = dt['rec_' + str(i) + '_' + j + '_suc_out_amt'] + dt['rec_' + str(i) + '_' + j + '_fail_out_amt']
            dt['rec_' + str(i) + '_' + j + '_fail_out_amtprop'] = dif_ser(dt['rec_' + str(i) + '_' + j + '_fail_out_amt'],t1)
            
            t2 = dt['rec_' + str(i) + '_' + j + '_suc_out_cnt'] + dt['rec_' + str(i) + '_' + j + '_fail_out_cnt']
            dt['rec_' + str(i) + '_' + j + '_fail_out_cntprop'] = dif_ser(dt['rec_' + str(i) + '_' + j + '_fail_out_cnt'],t2)
            
    
    
    %%time
    #机构数&新增机构数
    for i in date:
        for k in ['suc','fail']:
            for j in org_type3:        
                dt['rec_' + str(i) + '_' + j + '_' + k +'_out_ogn'] = dt['rec_' + str(i) + '_' + j + '_' + k + '_out_max'].apply(lambda x : len(x.keys()))
    
                if (j != 'all') & (i != 23):
                    s1 = dt['rec_' + str(i) + '_' + j + '_' + k + '_out_max']
                    dt_m = pd.DataFrame()
                    for m in range(i+1,24):
                        dt_m = pd.concat([dt_m,data['pre_' + str(m) + '_' + j + '_' + k + '_out_max']],axis = 1)
                    s2 = dt_m.apply(dict_max_dict,axis = 1) 
                    
                    #新增机构数
                    dt['rec_' + str(i) + '_' + j + '_' + k +'_out_newogn'] = new_org(s1,s2)[0]   
                    
                    #新增机构最大值
                    dt['rec_' + str(i) + '_' + j + '_' + k +'_out_newognmax'] = new_org(s1,s2)[1] 
                    
                    
            if i!= 23:
                dt['rec_' + str(i) + '_all_' + k +'_out_newogn'] = dt['rec_' + str(i) + '_bank_' +k +'_out_newogn'] + dt['rec_' + str(i) + '_cf_' +k +'_out_newogn'] + dt['rec_' + str(i) + '_top_' +k +'_out_newogn'] + dt['rec_' + str(i) + '_others_' +k +'_out_newogn']
                
                dt_new = pd.concat([dt['rec_' + str(i) + '_bank_' +k +'_out_newognmax'],dt['rec_' + str(i) + '_cf_' +k +'_out_newognmax'],dt['rec_' + str(i) + '_top_' +k +'_out_newognmax'],dt['rec_' + str(i) + '_others_' +k +'_out_newognmax']],axis = 1)
                dt['rec_' + str(i) + '_all_' + k +'_out_newognmax'] = dt_new.apply(dict_max_dict,axis = 1)
                              
    
    print('D')
    #基础变量时间占比
    dm = [1,3,6,12,23]
    for i in range(4):
        for j in range(i+1,5):
            for k in org_type3:
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_suc_out_cntprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_suc_out_cnt'],dt['rec_' + str(dm[j]) + '_' + k + '_suc_out_cnt'])
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_suc_out_amtprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_suc_out_amt'],dt['rec_' + str(dm[j]) + '_' + k + '_suc_out_amt'])
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_fail_out_cntprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_fail_out_cnt'],dt['rec_' + str(dm[j]) + '_' + k + '_fail_out_cnt'])
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_fail_out_amtprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_fail_out_amt'],dt['rec_' + str(dm[j]) + '_' + k + '_fail_out_amt'])
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_suc_in_cntprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_suc_in_cnt'],dt['rec_' + str(dm[j]) + '_' + k + '_suc_in_cnt'])                                                                                             
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_suc_in_amtprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_suc_in_amt'],dt['rec_' + str(dm[j]) + '_' + k + '_suc_in_amt'])                                                                                             
            dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_nonloan_suc_out_cntprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_nonloan_suc_out_cnt'],dt['rec_' + str(dm[j]) + '_nonloan_suc_out_cnt'])                                                                                             
            dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_nonloan_suc_out_amtprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_nonloan_suc_out_amt'],dt['rec_' + str(dm[j]) + '_nonloan_suc_out_amt'])                                                                                             
                                                                                                                          
    
    
    #机构&新增机构占比
    for i in range(4):
        for j in range(i+1,5):
            for k in org_type3:
                for m in ['suc','fail']:
                    dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_' + m + '_out_ognprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_' + m + '_out_ogn'],dt['rec_' + str(dm[j]) + '_' + k + '_' + m + '_out_ogn'])
                    if (i!= 3)&(j!=4):
                        dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_' + m + '_out_newognprop'] = dif_ser(dt['rec_' + str(dm[i]) + '_' + k + '_' + m + '_out_newogn'],dt['rec_' + str(dm[j]) + '_' + k + '_' + m + '_out_newogn'])
                    
                    
    
    
    #时间段金额差
    for i in range(4):
        for j in range(i+1,5):
            for k in org_type3:
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_suc_out_ad'] = dt['rec_' + str(dm[j]) + '_' + k + '_suc_out_amt']-dt['rec_' + str(dm[i]) + '_' + k + '_suc_out_amt']
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_fail_out_ad'] = dt['rec_' + str(dm[j]) + '_' + k + '_fail_out_amt'] - dt['rec_' + str(dm[i]) + '_' + k + '_fail_out_amt']
                dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_suc_in_ad'] = dt['rec_' + str(dm[j]) + '_' + k + '_suc_in_amt'] - dt['rec_' + str(dm[i]) + '_' + k + '_suc_in_amt']                                                                                            
            dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_nonloan_suc_out_ad'] = dt['rec_' + str(dm[j]) + '_nonloan_suc_out_amt'] - dt['rec_' + str(dm[i]) + '_nonloan_suc_out_amt']                                                                                             
    
    
    
    
    #时间段机构差
    for i in range(4):
        for j in range(i+1,5):
            for k in org_type3:
                for m in ['suc','fail']:
                    dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_' + m + '_out_od'] = dt['rec_' + str(dm[j]) + '_' + k + '_' + m + '_out_ogn'] - dt['rec_' + str(dm[i]) + '_' + k + '_' + m + '_out_ogn']
                    if (i!= 3)&(j!=4):
                        dt['r' + str(dm[i]) + '_' + str(dm[j]) + '_' + k + '_' + m + '_out_nod'] = dt['rec_' + str(dm[j]) + '_' + k + '_' + m + '_out_newogn'] - dt['rec_' + str(dm[i]) + '_' + k + '_' + m + '_out_newogn']
                    
     
    
    print('E')
    #成功转出-失败转出  & 成功转出-成功转入
    for i in date:
        for j in org_type3:
            dt['r' + str(i) + '_' + j + '_sf_out_cd'] = dt['rec_' + str(i) + '_' + j + '_suc_out_cnt']-dt['rec_' + str(i) + '_' + j + '_fail_out_cnt']
            dt['r' + str(i) + '_' + j + '_sf_out_ad'] = dt['rec_' + str(i) + '_' + j + '_suc_out_amt']-dt['rec_' + str(i) + '_' + j + '_fail_out_amt']
            dt['r' + str(i) + '_' + j + '_sf_out_mad'] = new_org(dt['rec_' + str(i) + '_' + j + '_suc_out_max'],dt['rec_' + str(i) + '_' + j + '_fail_out_max'])[1]
            
            dt['r' + str(i) + '_' + j + '_suc_oi_cd'] = dt['rec_' + str(i) + '_' + j + '_suc_out_cnt']-dt['rec_' + str(i) + '_' + j + '_suc_in_cnt']
            dt['r' + str(i) + '_' + j + '_suc_oi_ad'] = dt['rec_' + str(i) + '_' + j + '_suc_out_amt']-dt['rec_' + str(i) + '_' + j + '_suc_in_amt']
                 
    
    #最近交易时间&最早交易时间
    for i in org_type2:
        l1 = []
        l2 = []
        l3 = []
        for j in range(24):
            l1.append('pre_' + str(j) + '_' + i + '_suc_out_amt')
            l2.append('pre_' + str(j) + '_' + i + '_fail_out_amt')
            l3.append('pre_' + str(j) + '_' + i + '_suc_in_amt')
        #最近交易时间
        dt['late_' + i + '_suc_out'] = data[l1].apply(find_non_zero,axis = 1)
        dt['late_' + i + '_fail_out'] = data[l2].apply(find_non_zero,axis = 1)
        dt['late_' + i + '_suc_in'] = data[l3].apply(find_non_zero,axis = 1)
    
        #最早交易时间
        dt['early_' + i + '_suc_out'] = data[l1].apply(find_non_zero_back,axis = 1)
        dt['early_' + i + '_fail_out'] = data[l2].apply(find_non_zero_back,axis = 1)
        dt['early_' + i + '_suc_in'] = data[l3].apply(find_non_zero_back,axis = 1)
    
    #非银最近&非银最早
    l4 = []
    for i in range(24):
        l4.append('pre_' + str(i) + '_nonloan_suc_out_amt')
    dt['late_nonloan_suc_out'] = data[l4].apply(find_non_zero,axis = 1)
    dt['early_nonloan_suc_out'] = data[l4].apply(find_non_zero_back,axis = 1)
        
    #全部最近交易时间
    dt['late_all_suc_out'] = dt[['late_bank_suc_out','late_cf_suc_out','late_top_suc_out','late_others_suc_out',"late_nonloan_suc_out"]].apply(min,axis = 1)
    dt['late_all_fail_out'] = dt[['late_bank_fail_out','late_cf_fail_out','late_top_fail_out','late_others_fail_out']].apply(min,axis = 1)
    dt['late_all_suc_in'] = dt[['late_bank_suc_in','late_bank_suc_in','late_bank_suc_in','late_bank_suc_in']].apply(min,axis = 1)
    
    #全部最早交易时间
    dt['early_all_suc_out'] = dt[['early_bank_suc_out','early_cf_suc_out','early_top_suc_out','early_others_suc_out',"early_nonloan_suc_out"]].replace(99999, 0).apply(max,axis = 1)
    dt['early_all_fail_out'] = dt[['early_bank_fail_out','early_cf_fail_out','early_top_fail_out','early_others_fail_out']].replace(99999, 0).apply(max,axis = 1)
    dt['early_all_suc_in'] = dt[['early_bank_suc_in','early_bank_suc_in','early_bank_suc_in','early_bank_suc_in']].replace(99999, 0).apply(max,axis = 1)
    
    dt['early_all_suc_out'] = dt['early_all_suc_out'].replace(0,99999)
    dt['early_all_fail_out'] = dt['early_all_fail_out'].replace(0,99999)
    dt['early_all_suc_in'] = dt['early_all_suc_in'].replace(0,99999)
    
    
    #基础变量环比
    dm_inc = [1,3,6,12]
    org_type4 = ['bank','cf', 'top', 'others','all','nonloan']
    state_add = ['suc_out_cnt', 'suc_out_amt', 'suc_out_max', 'fail_out_cnt',
           'fail_out_amt', 'fail_out_max', 'suc_in_cnt', 'suc_in_amt']
    for i in dm_inc:
        for j in state_add:
            if j in ['suc_out_cnt', 'suc_out_amt']:
                for k in org_type4:
                    s1 = str(i) + '_' + k + '_' + j
                    if i != 12:
                        s2 = str(2*i) + '_' + k + '_' + j
                    else:
                        s2 = str(23) + '_' + k + '_' + j
                    t = dt['rec_' + s2] - dt['rec_' + s1]
                    dt['rec_' + s1 + 'inc'] = dif_ser(dt['rec_' + s1],t)
            elif j.split('_')[-1] != 'max':
                for k in org_type4[:-1]:
                    s1 = str(i) + '_' + k + '_' + j
                    if i != 12:
                        s2 = str(2*i) + '_' + k + '_' + j
                    else:
                        s2 = str(23) + '_' + k + '_' + j
                    t = dt['rec_' + s2] - dt['rec_' + s1]
                    dt['rec_' + s1 + 'inc'] = dif_ser(dt['rec_' + s1],t)
            else:
                for k in org_type4[:-1]:
                    s1 = str(i) + '_' + k + '_' + j
                    t1 = dt['rec_' + s1].apply(dict_max)
                    
                    dt_max = pd.DataFrame()
                    if k != 'all':
                        if i!=12:
                            for m in range(i+1,2*i+1):
                                ss = str(m) + '_' +k + '_'+j 
                                dt_max = pd.concat([dt_max,data['pre_' + ss]],axis = 1)
                        else:
                            for m in range(13,24):
                                ss = str(m) + '_' +k + '_'+j
                                dt_max = pd.concat([dt_max,data['pre_' + ss]],axis = 1)
                    else:
                        if i!=12:
                            for m in range(i+1,2*i+1):
                                ss = str(m) + '_' + j + '_all'
                                dt_max = pd.concat([dt_max,dt['rec_' + ss]],axis = 1)
                        else:
                            for m in range(13,24):
                                ss = str(m) + '_' + j + '_all'
                                dt_max = pd.concat([dt_max,dt['rec_' + ss]],axis = 1)
                    t2 = dt_max.apply(dict_max_num,axis = 1) 
                    dt['rec_' + s1 + 'inc'] = dif_ser(t1,t2)
    
    print('F')
    #机构环比
    for i in dm_inc:
        for j in org_type4[:-1]:
            for k in ['suc','fail']:
                s1 = str(i) + '_' + j + '_' + k + '_out_ogn'
                t1 = dt['rec_' + s1]
                
                dt_max = pd.DataFrame()
                if j != 'all':
                    if i != 12:
                        for m in range(i+1,2*i+1):
                            s2 = str(m) + '_' + j + '_' + k + '_out_max'
                            dt_max = pd.concat([dt_max,data['pre_' + s2]],axis = 1)
                        t2 = dt_max.apply(dict_max_dict,axis = 1)
                        
                    else:
                        for m in range(13,24):
                            s2 = str(m) + '_' + j + '_' + k + '_out_max'
                            dt_max = pd.concat([dt_max,data['pre_' + s2]],axis = 1)
                        t2 = dt_max.apply(dict_max_dict,axis = 1)
                else:                
                    if i != 12:
                        for m in range(i+1,2*i+1):
                            s2 = str(m) + '_' + k + '_out_max_all'
                            dt_max = pd.concat([dt_max,dt['rec_' + s2]],axis = 1)
                        t2 = dt_max.apply(dict_max_dict,axis = 1)
                    
                    else:
                        for m in range(13,24):
                            s2 = str(m) + '_' + k + '_out_max_all'
                            dt_max = pd.concat([dt_max,dt['rec_' + s2]],axis = 1)
                        t2 = dt_max.apply(dict_max_dict,axis = 1)
                t3 = t2.apply(lambda x : len(x.keys()))
                dt['rec_' + s1 + 'inc'] = dif_ser(t1,t3)
                
    
    
    #业务交叉
    for i in [0,1,3,6,12,18,23]:
        dt_bc = pd.DataFrame()
        for j in org_type2:
            lst = []        
            for k in ['_suc_out','_fail_out','_suc_in']:
                lst.append('rec_' +str(i) + '_' + j + k + '_amt')
            dt_bc[j] = dt[lst].apply(sum,axis = 1)
        t1 = (dt_bc[org_type2] != 0).sum(axis = 1)
        dt['r'+str(i) + '_bcto']  = t1.apply(lambda x :1 if x >= 2 else 0)
    
    
    
    #选取所需变量
    cols_all = pd.read_table('变量列表.csv',encoding = 'utf-8')
    cols_lst = list(cols_all['var_name'][1:])
    
    dt_all = dt[cols_lst]
    
    #将max列转为数值
    for i in dt_all.columns:
        if dt_all[i].dtype == 'O':
            dt_all[i] = dt_all[i].apply(lambda x :dict_max(x))     
    
    for i in [0, 1, 3, 6, 12, 18, 23]:
        for j in org_type3:
            dt_all['r'+str(i)+'_'+j+'_sf_out_mad'] = dt_all['rec_'+str(i)+'_'+j+'_suc_out_max']-dt_all['rec_'+str(i)+'_'+j+'_fail_out_max']
    
    dt_all_res = pd.concat([data[['index','idcard','dateBack','cardnum']],dt_all],axis = 1)
    #dt_all_res.rename(columns={'card':'cardnum'},inplace=True)
    
    
    # dt_all_res.to_csv('D:/实习/天创信用/11.29/Mock数据/百创指数_mock数据_1/dt_all_yes_ceshi.csv')
    
    col_1000_all = pd.read_csv('1000list.csv')
    col_1000_lst = list(col_1000_all['英文'])
    df_1000 = dt_all_res[['index','idcard','dateBack']+col_1000_lst]
    
    
    df_1000['fee_status'] = df_1000['cardnum'].apply(lambda x: 1 if x > 0 else 0)
    df_1000.to_csv(file_name + ".csv")

In [ ]:
get(df,"文件名.csv")
#del df
import gc
gc.collect()